In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

In [6]:
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "requirements.txt").exists():
    if (PROJECT_ROOT.parent / "requirements.txt").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "processed"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Artifacts directory: {ARTIFACTS_DIR}")

Project root: /Users/tolulopesoetan/Desktop/pl_prediction_model
Data directory: /Users/tolulopesoetan/Desktop/pl_prediction_model/data/processed
Artifacts directory: /Users/tolulopesoetan/Desktop/pl_prediction_model/artifacts


In [12]:
# load final feature set

ffeatures_v2 = pd.read_csv(DATA_DIR / "features_v2.csv")
features_v2["Date"] = pd.to_datetime(features_v2["Date"])
features_v2["season"] = features_v2["season"].astype(str)

print("Feature set shape:", features_v2.shape)
print("\nSeason counts:")
print(features_v2["season"].value_counts())
features_v2.head()

Feature set shape: (1504, 25)

Season counts:
season
2425    379
2324    378
2223    377
2122    370
Name: count, dtype: int64


,Date,HomeTeam,AwayTeam,FTR,FTHG,FTAG,HTHG,HTAG,home_form_pts,home_goals_for,...,away_goals_against,away_xg_for,away_xg_against,xg_diff,h2h_home_winrate,market_home_prob,market_draw_prob,market_away_prob,season,market_prediction
0,2021-08-21,Crystal Palace,Brentford,D,0,0,0,0,0.0,0.0,...,0.0,1.88818,1.02385,-1.566479,0.445395,0.372387,0.296746,0.330867,2122,H
1,2021-08-21,Liverpool,Burnley,H,2,0,1,0,3.0,3.0,...,2.0,1.79548,1.68530,-0.008200,0.445395,0.801216,0.126058,0.072726,2122,H
2,2021-08-21,Aston Villa,Newcastle,H,2,0,1,0,0.0,2.0,...,4.0,1.67926,3.00939,-0.542080,0.445395,0.527508,0.253204,0.219288,2122,H
3,2021-08-21,Man City,Norwich,H,5,0,2,0,0.0,0.0,...,3.0,1.33330,1.78728,0.772550,0.445395,0.877408,0.086146,0.036446,2122,H
4,2021-08-21,Brighton,Watford,H,2,0,2,0,3.0,2.0,...,2.0,1.35036,1.13718,0.334940,0.445395,0.561367,0.272664,0.165969,2122,H


In [13]:
# define pre-match features

FEATURES_V2 = [
    "home_form_pts",
    "home_goals_for",
    "home_goals_against",
    "home_xg_for",
    "home_xg_against",
    "away_form_pts",
    "away_goals_for",
    "away_goals_against",
    "away_xg_for",
    "away_xg_against",
    "xg_diff",
    "h2h_home_winrate",
    "market_home_prob",
    "market_draw_prob",
]

In [14]:
train = features_v2[features_v2["season"].isin(["2122","2223", "2324"])].copy()
test = features_v2[features_v2["season"] == "2425"].copy()

X_train = train[FEATURES_V2]
y_train = train["FTR"]
X_test = test[FEATURES_V2]
y_test = test["FTR"]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("\nTrain class distribution:")
print(y_train.value_counts(normalize=True))

Train shape: (1125, 14)
Test shape: (379, 14)

Train class distribution:
FTR
H    0.455111
A    0.317333
D    0.227556
Name: proportion, dtype: float64


In [16]:
# define helper functions

def multiclass_brier_score(y_true_labels, proba, class_order):
    """
    Compute the multiclass Brier score from true labels and predicted probabilities.

    The score is the mean squared difference between the one-hot encoded true outcome
    and the predicted probability vector for each row.
    """
    y_true_onehot = pd.get_dummies(pd.Categorical(y_true_labels, categories=class_order))
    y_true_onehot = y_true_onehot.reindex(columns=class_order, fill_value=0)
    return np.mean(np.sum((y_true_onehot.values - proba) ** 2, axis=1))


def evaluate_model(name, model, X_train, y_train, X_test, y_test, label_encoder, cv):
    """
    Fit a model, generate predictions and probabilities on the test set,
    and return a dictionary of evaluation metrics.

    The returned metrics include cross-validation accuracy, test accuracy,
    weighted F1, log-loss, and multiclass Brier score.
    """
    fitted_model = clone(model)
    fitted_model.fit(X_train, y_train)

    test_preds = fitted_model.predict(X_test)
    test_proba = fitted_model.predict_proba(X_test)

    cv_scores = cross_val_score(
        clone(model),
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy",
    )

    class_order = label_encoder.classes_

    return {
        "model": name,
        "cv_accuracy_mean": cv_scores.mean(),
        "test_accuracy": accuracy_score(y_test, test_preds),
        "weighted_f1": f1_score(y_test, test_preds, average="weighted"),
        "log_loss": log_loss(y_test, test_proba, labels=class_order),
        "brier_score": multiclass_brier_score(y_test, test_proba, class_order),
    }

In [17]:
# encode labels and define models

label_encoder = LabelEncoder()
label_encoder.fit(y_train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Logistic Regression": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
        ]
    ),
    "Gaussian Naive Bayes": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", GaussianNB()),
        ]
    ),
    "SVM": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", SVC(probability=True, class_weight="balanced", random_state=42)),
        ]
    ),
    "KNN": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier(n_neighbors=15)),
        ]
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6,
        min_samples_leaf=10,
        random_state=42,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=5,
        random_state=42,
    ),
}

In [18]:
# evaluate market baseline

market_proba = test[["market_home_prob", "market_draw_prob", "market_away_prob"]].copy()
market_proba.columns = ["H", "D", "A"]
market_proba = market_proba[label_encoder.classes_]

market_result = {
    "model": "Market (Bet365)",
    "cv_accuracy_mean": np.nan,
    "test_accuracy": accuracy_score(y_test, test["market_prediction"]),
    "weighted_f1": f1_score(y_test, test["market_prediction"], average="weighted"),
    "log_loss": log_loss(y_test, market_proba.values, labels=label_encoder.classes_),
    "brier_score": multiclass_brier_score(y_test, market_proba.values, label_encoder.classes_),
}

In [19]:
results = [market_result]

for model_name, model in models.items():
    result = evaluate_model(
        name=model_name,
        model=model,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        label_encoder=label_encoder,
        cv=cv,
    )
    results.append(result)

results_df = pd.DataFrame(results).sort_values("test_accuracy", ascending=False).reset_index(drop=True)
results_df

/Users/tolulopesoetan/Desktop/pl_prediction_model/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/tolulopesoetan/Desktop/pl_prediction_model/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/tolulopesoetan/Desktop/pl_prediction_model/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/tolulopesoetan/Desktop/pl_prediction_model/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: divide by zero encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ X + l2_reg_stren

,model,cv_accuracy_mean,test_accuracy,weighted_f1,log_loss,brier_score
0,Market (Bet365),NaN,0.538259,0.458940,0.972434,0.579865
1,Random Forest,0.564444,0.527704,0.452338,0.989357,0.590111
2,KNN,0.522667,0.525066,0.478439,1.568555,0.618376
3,Gaussian Naive Bayes,0.536000,0.519789,0.476105,1.584283,0.724197
4,Logistic Regression,0.510222,0.501319,0.497703,0.990743,0.594237
5,Decision Tree,0.503111,0.501319,0.466149,2.531822,0.627584
6,SVM,0.491556,0.490765,0.482787,0.999637,0.598057


In [20]:
results_df.round(4)

,model,cv_accuracy_mean,test_accuracy,weighted_f1,log_loss,brier_score
0,Market (Bet365),NaN,0.5383,0.4589,0.9724,0.5799
1,Random Forest,0.5644,0.5277,0.4523,0.9894,0.5901
2,KNN,0.5227,0.5251,0.4784,1.5686,0.6184
3,Gaussian Naive Bayes,0.5360,0.5198,0.4761,1.5843,0.7242
4,Logistic Regression,0.5102,0.5013,0.4977,0.9907,0.5942
5,Decision Tree,0.5031,0.5013,0.4661,2.5318,0.6276
6,SVM,0.4916,0.4908,0.4828,0.9996,0.5981


In [21]:
# save pre-match results

output_path = ARTIFACTS_DIR / "classical_model_results.csv"
results_df.to_csv(output_path, index=False)

print(f"Saved results to: {output_path}")

Saved results to: /Users/tolulopesoetan/Desktop/pl_prediction_model/artifacts/classical_model_results.csv


In [22]:
# build half time features

features_v2["ht_score_diff"] = features_v2["HTHG"] - features_v2["HTAG"]
features_v2["ht_home_goals"] = features_v2["HTHG"]
features_v2["ht_away_goals"] = features_v2["HTAG"]

features_v2["ht_home_winning"] = (features_v2["ht_score_diff"] > 0).astype(int)
features_v2["ht_away_winning"] = (features_v2["ht_score_diff"] < 0).astype(int)
features_v2["ht_drawing"] = (features_v2["ht_score_diff"] == 0).astype(int)

features_v2[
    [
        "HomeTeam",
        "AwayTeam",
        "HTHG",
        "HTAG",
        "ht_score_diff",
        "ht_home_winning",
        "ht_away_winning",
        "ht_drawing",
    ]
].head()

,HomeTeam,AwayTeam,HTHG,HTAG,ht_score_diff,ht_home_winning,ht_away_winning,ht_drawing
0,Crystal Palace,Brentford,0,0,0,0,0,1
1,Liverpool,Burnley,1,0,1,1,0,0
2,Aston Villa,Newcastle,1,0,1,1,0,0
3,Man City,Norwich,2,0,2,1,0,0
4,Brighton,Watford,2,0,2,1,0,0


In [23]:
# define half-time feature set

FEATURES_HT = FEATURES_V2 + [
    "ht_score_diff",
    "ht_home_goals",
    "ht_away_goals",
    "ht_home_winning",
    "ht_away_winning",
    "ht_drawing",
]

In [25]:
train_ht = features_v2[features_v2["season"].isin(["2122","2223", "2324"])].copy()
test_ht = features_v2[features_v2["season"] == "2425"].copy()

X_train_ht = train_ht[FEATURES_HT]
y_train_ht = train_ht["FTR"]
X_test_ht = test_ht[FEATURES_HT]
y_test_ht = test_ht["FTR"]

print("Half-time train shape:", X_train_ht.shape)
print("Half-time test shape:", X_test_ht.shape)

Half-time train shape: (1125, 20)
Half-time test shape: (379, 20)


In [26]:
# compare pre-match and half-time models

comparison_models = {
    "LR - pre-match": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
        ]
    ),
    "LR - half-time": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
        ]
    ),
    "RF - pre-match": RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=5,
        random_state=42,
    ),
    "RF - half-time": RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=5,
        random_state=42,
    ),
}

In [27]:
ht_results = []

ht_results.append(
    evaluate_model(
        name="LR - pre-match",
        model=comparison_models["LR - pre-match"],
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        label_encoder=label_encoder,
        cv=cv,
    )
)

ht_results.append(
    evaluate_model(
        name="LR - half-time",
        model=comparison_models["LR - half-time"],
        X_train=X_train_ht,
        y_train=y_train_ht,
        X_test=X_test_ht,
        y_test=y_test_ht,
        label_encoder=label_encoder,
        cv=cv,
    )
)

ht_results.append(
    evaluate_model(
        name="RF - pre-match",
        model=comparison_models["RF - pre-match"],
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        label_encoder=label_encoder,
        cv=cv,
    )
)

ht_results.append(
    evaluate_model(
        name="RF - half-time",
        model=comparison_models["RF - half-time"],
        X_train=X_train_ht,
        y_train=y_train_ht,
        X_test=X_test_ht,
        y_test=y_test_ht,
        label_encoder=label_encoder,
        cv=cv,
    )
)

ht_df = pd.DataFrame(ht_results).sort_values("test_accuracy", ascending=False).reset_index(drop=True)
ht_df.round(4)

/Users/tolulopesoetan/Desktop/pl_prediction_model/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/tolulopesoetan/Desktop/pl_prediction_model/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/tolulopesoetan/Desktop/pl_prediction_model/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/tolulopesoetan/Desktop/pl_prediction_model/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: divide by zero encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ X + l2_reg_stren

,model,cv_accuracy_mean,test_accuracy,weighted_f1,log_loss,brier_score
0,RF - half-time,0.6258,0.6095,0.5741,0.8837,0.5148
1,LR - half-time,0.5991,0.5805,0.5855,0.8591,0.5093
2,RF - pre-match,0.5644,0.5277,0.4523,0.9894,0.5901
3,LR - pre-match,0.5102,0.5013,0.4977,0.9907,0.5942


In [28]:
# save half-time comparison results

ht_output_path = ARTIFACTS_DIR / "halftime_model_results.csv"
ht_df.to_csv(ht_output_path, index=False)

print(f"Saved half-time results to: {ht_output_path}")

Saved half-time results to: /Users/tolulopesoetan/Desktop/pl_prediction_model/artifacts/halftime_model_results.csv


In [29]:
# export test-set predictions for evaluation notebook

def fit_and_predict(model, X_train, y_train, X_test, class_order):
    """
    Fit a model and return predicted classes and class probabilities on the test set.

    The returned probability dataframe is ordered according to class_order.
    """
    fitted_model = clone(model)
    fitted_model.fit(X_train, y_train)

    preds = fitted_model.predict(X_test)
    proba = pd.DataFrame(
        fitted_model.predict_proba(X_test),
        columns=fitted_model.classes_,
        index=X_test.index,
    ).reindex(columns=class_order, fill_value=0.0)

    return preds, proba

In [30]:
lr_pre_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
    ]
)

lr_ht_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
    ]
)

rf_pre_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=5,
    random_state=42,
)

rf_ht_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=5,
    random_state=42,
)

class_order = list(label_encoder.classes_)
print("Class order:", class_order)

Class order: ['A', 'D', 'H']


In [31]:
lr_pre_preds, lr_pre_proba = fit_and_predict(
    lr_pre_model, X_train, y_train, X_test, class_order
)
lr_ht_preds, lr_ht_proba = fit_and_predict(
    lr_ht_model, X_train_ht, y_train_ht, X_test_ht, class_order
)
rf_pre_preds, rf_pre_proba = fit_and_predict(
    rf_pre_model, X_train, y_train, X_test, class_order
)
rf_ht_preds, rf_ht_proba = fit_and_predict(
    rf_ht_model, X_train_ht, y_train_ht, X_test_ht, class_order
)

/Users/tolulopesoetan/Desktop/pl_prediction_model/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/tolulopesoetan/Desktop/pl_prediction_model/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/tolulopesoetan/Desktop/pl_prediction_model/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/tolulopesoetan/Desktop/pl_prediction_model/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: divide by zero encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ X + l2_reg_stren

In [32]:
test_predictions = test.copy().reset_index(drop=True)

market_export = test_predictions[
    ["market_home_prob", "market_draw_prob", "market_away_prob", "market_prediction"]
].copy()

test_predictions_export = pd.DataFrame(
    {
        "Date": test_predictions["Date"],
        "HomeTeam": test_predictions["HomeTeam"],
        "AwayTeam": test_predictions["AwayTeam"],
        "FTR": test_predictions["FTR"],
        "market_home_prob": market_export["market_home_prob"],
        "market_draw_prob": market_export["market_draw_prob"],
        "market_away_prob": market_export["market_away_prob"],
        "market_prediction": market_export["market_prediction"],
        "lr_pre_pred": lr_pre_preds,
        "lr_ht_pred": lr_ht_preds,
        "rf_pre_pred": rf_pre_preds,
        "rf_ht_pred": rf_ht_preds,
    }
)

In [33]:
for cls in class_order:
    test_predictions_export[f"lr_pre_prob_{cls}"] = lr_pre_proba[cls].values
    test_predictions_export[f"lr_ht_prob_{cls}"] = lr_ht_proba[cls].values
    test_predictions_export[f"rf_pre_prob_{cls}"] = rf_pre_proba[cls].values
    test_predictions_export[f"rf_ht_prob_{cls}"] = rf_ht_proba[cls].values

test_predictions_export.head()

,Date,HomeTeam,AwayTeam,FTR,market_home_prob,market_draw_prob,market_away_prob,market_prediction,lr_pre_pred,lr_ht_pred,...,rf_pre_prob_A,rf_ht_prob_A,lr_pre_prob_D,lr_ht_prob_D,rf_pre_prob_D,rf_ht_prob_D,lr_pre_prob_H,lr_ht_prob_H,rf_pre_prob_H,rf_ht_prob_H
0,2024-08-16,Man United,Fulham,H,0.593220,0.225989,0.180791,H,H,D,...,0.150599,0.207645,0.372491,0.479944,0.171122,0.277290,0.484251,0.394933,0.678279,0.515066
1,2024-08-17,Arsenal,Wolves,H,0.801216,0.126058,0.072726,H,H,H,...,0.030195,0.004454,0.107939,0.084130,0.099441,0.094463,0.840025,0.899064,0.870363,0.901083
2,2024-08-17,Everton,Brighton,A,0.357530,0.284940,0.357530,H,A,A,...,0.383259,0.819808,0.372931,0.273639,0.232405,0.128287,0.229064,0.070982,0.384336,0.051906
3,2024-08-17,Newcastle,Southampton,H,0.699767,0.181273,0.118960,H,H,H,...,0.144943,0.009076,0.188538,0.166590,0.175440,0.158586,0.692981,0.783421,0.679618,0.832339
4,2024-08-17,Nott'm Forest,Bournemouth,D,0.388350,0.271845,0.339806,H,A,H,...,0.306547,0.062363,0.325165,0.333886,0.270296,0.300861,0.270745,0.488085,0.423157,0.636777


In [34]:
predictions_output_path = ARTIFACTS_DIR / "test_predictions.csv"
test_predictions_export.to_csv(predictions_output_path, index=False)

print(f"Saved exported test predictions to: {predictions_output_path}")

Saved exported test predictions to: /Users/tolulopesoetan/Desktop/pl_prediction_model/artifacts/test_predictions.csv
